# Pixelle-Video · Flexible Video (generate-or-search) × Wan2GP on Google Colab

Turns a **topic** into a narrated short video where an **LLM orchestrator decides per scene** whether the visual is **AI-generated** (WanGP, in-process) or **real stock media** searched on **Pexels / Pixabay** — with review/edit checkpoints between every stage:

```
① Script    topic → video title + scene narrations            →  you edit / ✨AI-rewrite
② Plan      LLM media plan per scene: 🎨 generate vs 🔎 search →  you override / edit
③ Source    search scenes: stock candidates → LLM auto-pick   →  you override the pick
            → download + ffmpeg-normalize into the scene assets
④ Scenes    per scene: 🎤 audio → 🖼️/🎬 media → 🎞️ segment
⑤ Final     compose all segments (+ optional BGM) → final.mp4 + 📜 stock credits
```

Three things make the flexible pipeline special:
- **LLM orchestration** — real-world concepts (nature, cities, people, b-roll) get real stock footage; abstract/stylized concepts get AI generation. You can override every decision.
- **Quota-friendly search** — Pexels (200 req/h, 20k/mo) and Pixabay (100 req/60s) responses are cached on disk for 24h (a Pixabay API requirement) and picked media is always downloaded + re-encoded locally (no hotlinking).
- **Concat-safe normalization** — stock clips are re-encoded to the project size/fps (cover-crop, yuv420p, silent) so they mix freely with generated scenes in the final video.

Media is generated with the **Wan2GP in-process backend** (models loaded directly in this runtime through WanGP's Python API). No ComfyUI server, no RunningHub key.

Run the cells in order. Dependencies are installed **once** and shared by everything.

> **Colab VRAM note:** the free tier usually assigns a 15 GB T4 GPU. The defaults below are sized for it: **Z-Image Turbo 6B** for images and **Wan 2.1 1.3B** for video clips. On a bigger GPU (L4/A100) switch to `wan2gp/image_qwen.json`, `wan2gp/video_wan2.1_fusionx.json` or `wan2gp/video_ltx2_distilled.json`.

> **LLM note:** Pixelle-Video needs an OpenAI-compatible LLM endpoint (Qwen/DashScope, DeepSeek, OpenAI, Ollama, your own vLLM tunnel, ...) to write the script, plan the media sources and rank stock candidates. Fill it in at step 6.

> **Stock API keys (optional but recommended):** get free keys at [pexels.com/api](https://www.pexels.com/api/) and [pixabay.com/api/docs](https://pixabay.com/api/docs/) and fill them in at step 6. **With no keys the notebook still works** — every scene is simply AI-generated.


## 1. Confirm the accelerator

Choose `Runtime → Change runtime type` and select **GPU** before running anything else.

If this cell raises an error, go back to `Runtime → Change runtime type`, pick **GPU** and save.


In [ ]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. In Colab, open Runtime -> Change runtime type, select GPU, save, then rerun this cell.'
    ) from exc


## 2. Configure the workspace path

Choose where Wan2GP should be installed. It bundles all the Pixelle-Video flavors:
- `Pixelle_video/` — the original fully-automatic app (also holds the shared config / workflows / templates / output)
- `Pixelle_video_scene_by_scene/` — the step-by-step engine this pipeline builds on
- `Pixelle_video_flexible/` — the generate-or-search engine used by this notebook


In [ ]:
from pathlib import Path

WAN2GP_ROOT  = Path('/content/wan2gp').resolve()
PIXELLE_ROOT = WAN2GP_ROOT / 'Pixelle_video'
SBS_ROOT     = WAN2GP_ROOT / 'Pixelle_video_scene_by_scene'
FLEX_ROOT    = WAN2GP_ROOT / 'Pixelle_video_flexible'
print(f'Wan2GP will be installed to:   {WAN2GP_ROOT}')
print(f'Pixelle-Video (core + data):   {PIXELLE_ROOT}')
print(f'Scene-by-Scene engine:         {SBS_ROOT}')
print(f'Flexible (gen-or-search) app:  {FLEX_ROOT}')


## 3. Download or update Wan2GP

Clone the repository on the `flexible-material-search-Pixelle` branch (the one that contains `Pixelle_video_flexible/`); pull the latest changes if it already exists.


In [ ]:
import subprocess

repo_url = 'https://github.com/hoangthvn2201/Wan2GP'
branch = 'flexible-material-search-Pixelle'   # change to 'main' once the branch is merged

if WAN2GP_ROOT.exists():
    print('Repository already exists. Updating...')
    subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', branch], check=True)
subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull', 'origin', branch], check=True)


## 4. Install system dependencies

Shared libraries for video and audio processing. If you see a warning about skipping an extra repository, it is safe to ignore.


In [ ]:
import os, subprocess

env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'

subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
subprocess.run([
    'sudo', 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'
], check=True, env=env)


## 5. Install Python dependencies (one-time)

A **single install** covers everything:

- `requirements.txt` (Wan2GP) — torch ecosystem, diffusers, loguru, pydantic, moviepy, ffmpeg-python, ...
- `Pixelle_video/requirements.txt` — the Pixelle extras (streamlit, openai, edge-tts, comfykit, playwright, ...)

The flexible app adds **no Python dependencies of its own** (stock search uses `httpx`, already a core requirement). Afterwards Chromium is installed for the HTML frame-template rendering. This cell takes several minutes.


In [ ]:
import os, subprocess, sys

env = os.environ.copy()
env.setdefault('DEBIAN_FRONTEND', 'noninteractive')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True, env=env)

subprocess.run([sys.executable, '-m', 'pip', 'install',
    '--force-reinstall', '--no-deps',
    'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True, env=env)

subprocess.run([sys.executable, '-m', 'pip', 'install', 'xformers==0.0.32.post2',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True, env=env)

# One-time install: Wan2GP requirements + Pixelle-Video extras in a single resolve
# (the flexible app has no extra deps of its own)
subprocess.run([sys.executable, '-m', 'pip', 'install',
    '-r', str(WAN2GP_ROOT / 'requirements.txt'),
    '-r', str(PIXELLE_ROOT / 'requirements.txt')], check=True, env=env)

# Chromium for HTML frame template rendering (Pixelle composes subtitles via Playwright)
subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True, env=env)

# Re-assert a modern setuptools AFTER all installs: the dependency resolve can
# remove it from /usr/local, letting Ubuntu's ancient system pkg_resources
# (which still uses pkgutil.ImpImporter, removed in Python 3.12) shadow it.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'setuptools', 'wheel'], check=True, env=env)


## 5b. Force a headless matplotlib backend

Ensure Wan2GP's preprocessing tools use the headless Agg backend.


In [ ]:
from pathlib import Path

target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"

if not target.exists():
    print(f'Skipping: {target} not found.')
else:
    text = target.read_text()
    if replacement in text:
        print('Agg backend already set; no change needed.')
    elif needle in text:
        target.write_text(text.replace(needle, replacement, 1))
        print('Replaced TkAgg with Agg in interact_tools.py.')
    else:
        print('Backend call not found; no change made.')


## 5c. (Optional) Vietnamese TTS — VieNeu-TTS

**Skip this cell** unless you want on-device Vietnamese narration via [VieNeu-TTS](https://github.com/pnnbao97/VieNeu-TTS).

It installs *on top of* the standard environment and does **not** change any package pinned in step 5 (`vieneu` itself is installed `--no-deps` — see `Pixelle_video/requirements-vieneu.txt` for why). Expect **~10–15 minutes**: `llama-cpp-python` compiles from source (cmake/gcc are preinstalled on Colab) and `sea-g2p` needs a Rust toolchain (installed automatically below). Model weights (GGUF + ONNX codec, ~1 GB) are downloaded from HuggingFace on the first synthesis.

Afterwards set `TTS_MODE = 'vieneu'` in step 6. Preset voices (fuzzy-matched): `Xuân Vĩnh (Nam - Miền Nam)`, `Phạm Tuyên (Nam - Miền Bắc)`, `Bích Ngọc (Nữ - Miền Bắc)`, `Thục Đoan (Nữ - Miền Nam)` — e.g. `TTS_VOICE = 'Bích Ngọc'` works.

> Tip: pair it with `NARRATION_LANGUAGE = 'Vietnamese'` in step 8 to make a Vietnamese video about an English-language topic.

**Quality tiers** (`TTS_VIENEU_MODE` in step 6): `max` = full-precision VieNeu-TTS-v2 backbone + full NeuCodec (best quality + high-fidelity voice cloning with `ref_text`, runs on the GPU, ~1.6 GB extra weights) — its extras are installed by this cell; `standard` = Q4 GGUF + int8 ONNX (good, lighter, no cloning); `turbo` = fastest, lowest quality. A tier whose dependencies are missing automatically falls back to the next one, so this never breaks the pipeline.


In [ ]:
# ⏭️ OPTIONAL — only for Vietnamese narration (VieNeu-TTS). Safe to skip.
import os, shutil, subprocess, sys

env = os.environ.copy()
cargo_bin = os.path.expanduser('~/.cargo/bin')

# 1. Rust toolchain for sea-g2p (no Linux x86_64 wheel on PyPI -> builds via maturin)
if shutil.which('cargo') is None and not os.path.exists(f'{cargo_bin}/cargo'):
    subprocess.run('curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal',
                   shell=True, check=True)
env['PATH'] = f"{cargo_bin}:{env['PATH']}"

# 2. VieNeu runtime deps (llama-cpp-python compiles from source: ~10 min)
subprocess.run([sys.executable, '-m', 'pip', 'install',
    '-r', str(PIXELLE_ROOT / 'requirements-vieneu.txt')], check=True, env=env)

# 3. vieneu itself WITHOUT its declared deps: they would clobber the Wan2GP
#    environment from step 5 (gradio>=5.49.1 vs 5.29.0, CPU onnxruntime vs
#    onnxruntime-gpu). Everything it actually needs is already installed.
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'vieneu==2.7.0', '--no-deps'], check=True, env=env)

# 4. OPTIONAL max-quality extras (TTS_VIENEU_MODE = 'max' in step 6: full-precision
#    VieNeu-TTS-v2 backbone + full NeuCodec — best audio quality + voice cloning).
#    neucodec must be --no-deps: it pins vector-quantize-pytorch==1.17.8 and torch
#    versions that would clobber the Wan2GP env (see requirements-vieneu.txt).
#    If this step fails, VieNeu still works — 'max' falls back to 'standard'.
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install',
        'torchao', 'torchtune', 'local-attention'], check=True, env=env)
    subprocess.run([sys.executable, '-m', 'pip', 'install',
        'neucodec==0.0.5', '--no-deps'], check=True, env=env)
    max_ok = True
except subprocess.CalledProcessError as e:
    max_ok = False
    print(f"⚠️ Max-quality extras failed ({e}); 'max' will fall back to 'standard'.")

print("✅ VieNeu-TTS installed — set TTS_MODE = 'vieneu' in step 6 "
      "(voices: Xuân Vĩnh, Phạm Tuyên, Bích Ngọc, Thục Đoan — fuzzy-matched)."
      + (" Max-quality extras ready: keep TTS_VIENEU_MODE = 'max'." if max_ok else ""))


## 6. Configure Pixelle-Video + the stock providers

Fill in your **LLM endpoint** (required) and your **Pexels / Pixabay API keys** (optional — both free), then run the cell. It writes:
- `Pixelle_video/config.yaml` — the shared core config (LLM, workflows, TTS)
- `Pixelle_video_flexible/flex_config.yaml` — the media-search config

The `wan2gp/...` workflows are *descriptors* that map to WanGP models (see `Pixelle_video/WAN2GP_BACKEND.md`). Model checkpoints are downloaded automatically by WanGP on first use.

> **No stock keys?** Leave them empty — the media plan then sets every scene to AI generation and everything still works.


In [ ]:
import yaml

# --- LLM (required: writes the script, plans sources, ranks candidates) -----
LLM_API_KEY  = ''                                  # <-- your API key
LLM_BASE_URL = 'https://api.deepseek.com'          # any OpenAI-compatible endpoint
LLM_MODEL    = 'deepseek-chat'

# --- Stock media search (optional, both free) --------------------------------
PEXELS_API_KEY  = ''               # https://www.pexels.com/api/      (200 req/h, 20k/mo)
PIXABAY_API_KEY = ''               # https://pixabay.com/api/docs/    (100 req/60s)
CANDIDATES_PER_SCENE = 6           # stock candidates fetched/shown per scene
MIN_RESOLUTION = 720               # discard results whose short side is below this
ALLOW_FALLBACK = True              # empty search results -> fall back to generation
SEARCH_ONLY = False                # True = stock-only: every scene searches, empty
                                   # results retry with broader keywords, AI generation
                                   # is NEVER used (no model download, no GPU needed)

# --- Media generation (wan2gp = models loaded in-process by WanGP) ---------
IMAGE_WORKFLOW = 'wan2gp/image_z_image.json'       # Z-Image Turbo 6B  (T4-friendly)
VIDEO_WORKFLOW = 'wan2gp/video_wan2.1_1.3B.json'   # Wan 2.1 1.3B t2v  (T4-friendly)
# Bigger GPUs:
#   IMAGE_WORKFLOW = 'wan2gp/image_qwen.json'             # Qwen Image 20B
#   VIDEO_WORKFLOW = 'wan2gp/video_wan2.1_fusionx.json'   # Wan 2.1 FusioniX 14B
#   VIDEO_WORKFLOW = 'wan2gp/video_ltx2_distilled.json'   # LTX-2 22B (video + audio)

PROMPT_PREFIX = ('Minimalist black-and-white matchstick figure style illustration, '
                 'clean lines, simple sketch style')
# Tip: a strong style prefix makes GENERATED scenes look very different from
# real STOCK footage. For a mixed gen+stock video consider a photographic
# prefix (e.g. 'cinematic photo, natural light, shallow depth of field') or
# an empty PROMPT_PREFIX = '' so both kinds of scenes blend together.

# --- TTS ---------------------------------------------------------------------
TTS_MODE  = 'local'              # 'local' = edge-tts (online, free)
                                 # 'vieneu' = Vietnamese on-device (run cell 5c first!)
TTS_VOICE = 'en-US-GuyNeural'    # local: e.g. 'zh-CN-YunjianNeural' for Chinese
                                 # vieneu: 'Xuân Vĩnh', 'Phạm Tuyên', 'Bích Ngọc', 'Thục Đoan' (fuzzy-matched)
TTS_SPEED = 1.2                  # vieneu tip: 1.0 sounds most natural
TTS_VIENEU_MODE = 'max'          # vieneu quality: 'max' (best; needs the extras from
                                 # cell 5c) | 'standard' | 'turbo' (fastest, lowest
                                 # quality). Missing deps auto-fall back one tier.
TTS_NORMALIZE = True             # loudness-normalize the narration to -16 LUFS (fixes quiet TTS)
TTS_VOLUME = 1.0                 # extra gain on top (1.0 = unchanged, 1.5 = +50%; clip-guarded)

# --- WanGP runtime -----------------------------------------------------------
WAN2GP_CLI_ARGS = ['--profile', '5']   # low-VRAM profile (same as the official notebook)

config = {
    'project_name': 'Pixelle-Video',
    'llm': {
        'api_key': LLM_API_KEY,
        'base_url': LLM_BASE_URL,
        'model': LLM_MODEL,
        'enable_thinking': False,
    },
    'comfyui': {
        'comfyui_url': 'http://127.0.0.1:8188',
        'comfyui_api_key': None,
        'runninghub_api_key': None,
        'runninghub_concurrent_limit': 1,
        'runninghub_instance_type': None,
        'tts': {
            'inference_mode': TTS_MODE,
            'local': {'voice': TTS_VOICE if TTS_MODE == 'local' else 'zh-CN-YunjianNeural',
                      'speed': TTS_SPEED if TTS_MODE == 'local' else 1.2},
            'vieneu': {'voice': TTS_VOICE if TTS_MODE == 'vieneu' else 'Xuân Vĩnh (Nam - Miền Nam)',
                       'speed': TTS_SPEED if TTS_MODE == 'vieneu' else 1.0,
                       'ref_audio': None, 'ref_text': None,
                       'mode': TTS_VIENEU_MODE, 'device': 'cuda'},
            'comfyui': {'default_workflow': None},
        },
        'image': {'default_workflow': IMAGE_WORKFLOW, 'prompt_prefix': PROMPT_PREFIX},
        'video': {'default_workflow': VIDEO_WORKFLOW, 'prompt_prefix': PROMPT_PREFIX},
    },
    'wan2gp': {
        'root': str(WAN2GP_ROOT),
        'cli_args': WAN2GP_CLI_ARGS,
        'output_dir': None,
    },
    'template': {'default_template': '1080x1920/image_default.html'},
}

(PIXELLE_ROOT / 'config.yaml').write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
print('config.yaml written.')

flex_config = {
    'media_search': {
        'pexels':  {'enabled': True, 'api_key': PEXELS_API_KEY},
        'pixabay': {'enabled': True, 'api_key': PIXABAY_API_KEY},
        'candidates_per_scene': CANDIDATES_PER_SCENE,
        'min_resolution': MIN_RESOLUTION,
        'allow_fallback': ALLOW_FALLBACK,
        'search_only': SEARCH_ONLY,
    }
}
(FLEX_ROOT / 'flex_config.yaml').write_text(yaml.safe_dump(flex_config, allow_unicode=True, sort_keys=False))
print('flex_config.yaml written.')
if SEARCH_ONLY:
    print('🔒 Stock-only mode: all scenes will use stock search — no generation model is ever loaded.')

providers = [name for name, key in (('pexels', PEXELS_API_KEY), ('pixabay', PIXABAY_API_KEY)) if key.strip()]
if providers:
    print(f"🔎 Stock search enabled: {', '.join(providers)}")
else:
    print('⚠️ No stock API keys — every scene will be AI-generated (the notebook still works).')


## 7. Initialize the core + the Flexible engine

Sets up paths, initializes the core services and creates the `FlexibleVideoEngine` (it extends the scene-by-scene engine, so all per-scene generation steps are shared). The WanGP session itself is created lazily — model weights are only loaded (and downloaded) on the first generation.


In [ ]:
import os, sys

os.chdir(PIXELLE_ROOT)                                   # relative paths: workflows/, templates/, output/
os.environ['PIXELLE_VIDEO_ROOT'] = str(PIXELLE_ROOT)

# sys.path: flexible app (flexvid) + scene-by-scene engine (sbs) + core + Wan2GP root
for p in (str(WAN2GP_ROOT), str(PIXELLE_ROOT), str(SBS_ROOT), str(FLEX_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

from pixelle_video.service import pixelle_video
from flexvid import FlexibleVideoEngine

await pixelle_video.initialize()
engine = FlexibleVideoEngine(pixelle_video)              # reads flex_config.yaml
print(pixelle_video)
print('wan2gp media workflows:', [w for w in pixelle_video.media.available if w.startswith('wan2gp/')])
print('stock providers:       ', engine.aggregator.provider_names or 'none (all scenes will be generated)')


## 8. Step ① — Generate the script (then review it)

The LLM writes the video title plus one narration per scene from your topic.

> Template naming controls the media type: `image_*.html` templates use the image model, `video_*.html` templates use the video model. **For a mixed gen+stock video the `video_*` template is the most flexible** — search scenes can then use real stock *clips* (with stock photos still allowed per scene), while generate scenes use the video model.
>
> `NARRATION_LANGUAGE = None` keeps the topic's own language; set e.g. `'Vietnamese'` to narrate in another language (pair with the VieNeu voice from cell 5c).


In [ ]:
TOPIC    = 'Why the ocean matters more than we think'
N_SCENES = 5
TEMPLATE = '1080x1920/video_default.html'   # 'image_*.html' for stills-only scenes
NARRATION_LANGUAGE = None                   # None = the topic's language; or 'Vietnamese', 'English', ...

from pixelle_video.utils.template_util import resolve_template_path, get_template_type
from pixelle_video.services.frame_html import HTMLFrameGenerator

template_type = get_template_type(TEMPLATE.split('/')[-1])    # 'static' | 'image' | 'video'
media_workflow = {'image': IMAGE_WORKFLOW, 'video': VIDEO_WORKFLOW}.get(template_type)
media_width, media_height = HTMLFrameGenerator(resolve_template_path(TEMPLATE)).get_media_size()
print(f'Template type: {template_type} | media workflow: {media_workflow} | media size: {media_width}x{media_height}')
print()

script_topic = TOPIC if not NARRATION_LANGUAGE else f'{TOPIC}\n\n(Write the narration in {NARRATION_LANGUAGE}.)'
title, narrations = await engine.generate_script(script_topic, mode='generate', n_scenes=N_SCENES)

print(f'Title: {title}')
for i, n in enumerate(narrations, 1):
    print(f'  Scene {i}: {n}')


### Edit the script (optional)

Adjust anything before moving on:

- **manual edit**: assign directly into the `narrations` list (or change `title`)
- **✨ AI rewrite**: ask the LLM to rewrite one scene, optionally with an instruction
- **add / remove scenes**: regular Python list operations


In [ ]:
# --- Manual edits (uncomment & adapt) ---------------------------------------
# title = 'My better title'
# narrations[0] = 'My own opening line for scene 1.'
# narrations.append('One extra closing scene.')
# del narrations[2]

# --- AI rewrite of a single scene (uncomment to use) -------------------------
# narrations[0] = await engine.rewrite_narration(
#     narrations[0], topic=TOPIC,
#     instruction='make it a question that hooks the viewer',
# )

print(f'Title: {title}')
for i, n in enumerate(narrations, 1):
    print(f'  Scene {i}: {n}')


## 9. Create the project

Freezes the script + settings into a `SceneProject` with its own task directory (`Pixelle_video/output/<task_id>/`). Each scene is a `FlexScene` with a stable `uid`, carrying its media-sourcing plan alongside the usual assets.


In [ ]:
params = {
    'text': TOPIC,
    'mode': 'generate',
    'n_scenes': len(narrations),
    'title': title,
    'tts_inference_mode': TTS_MODE,
    'tts_voice': TTS_VOICE,
    'tts_speed': TTS_SPEED,
    'tts_normalize': TTS_NORMALIZE,   # post-TTS loudness normalization (engine-applied)
    'tts_volume': TTS_VOLUME,         # extra narration gain on top
    'frame_template': TEMPLATE,
    'template_params': None,
    'media_workflow': media_workflow,
    'prompt_prefix': PROMPT_PREFIX,
    'media_width': media_width,
    'media_height': media_height,
    # --- Flexible provenance (persisted with the task) ---
    'flex_language': NARRATION_LANGUAGE,
    'flex_providers': engine.aggregator.provider_names,
}

project = engine.create_project(title, narrations, [None] * len(narrations), params)
print(f'Task: {project.task_id}  ({len(project.scenes)} scenes, media={project.media_requirement})')
print(f'Dir:  {project.task_dir}')


## 10. Step ② — Plan the media sources (then review the plan)

The LLM decides **per scene**: 🔎 **search** real stock media (with short English keywords) or 🎨 **generate** it (with a full media prompt). Real-world/b-roll concepts lean search; abstract/stylized concepts lean generate. With no providers configured, every scene is set to generate; with `SEARCH_ONLY = True` (step 6), every scene is set to search and abstract ideas are translated into concrete stock-findable imagery.


In [ ]:
await engine.generate_scene_plan(project, prompt_prefix=PROMPT_PREFIX)

for i, sc in enumerate(project.scenes, 1):
    icon = '🔎' if sc.source == 'search' else '🎨'
    print(f'Scene {i}  {icon} {sc.source.upper()}  [{sc.plan_media_type}]')
    print(f'   🗣️ {sc.narration}')
    if sc.source == 'search':
        print(f'   🔎 keywords: {sc.search_query}')
    else:
        print(f'   🎨 prompt:   {sc.prompt}')
    if sc.plan_reason:
        print(f'   🤖 {sc.plan_reason}')
    print()


### Edit the plan (optional)

Override any decision before sourcing — flip a scene between search/generate, tune the keywords or prompt, or let the AI re-plan a single scene:


In [ ]:
# --- Flip a scene's source (uncomment & adapt) --------------------------------
# sc = project.scenes[0]
# sc.source = 'search'; sc.search_query = 'waves crashing rocky coast'
# sc.plan_media_type = 'video'             # or 'image' for a stock photo
# sc.invalidate_search(); sc.invalidate_media()

# sc = project.scenes[1]
# sc.source = 'generate'
# sc.prompt = 'a glowing blue whale drifting through a starry abyss, cinematic, ' + PROMPT_PREFIX
# sc.invalidate_search(); sc.invalidate_media()

# --- Or let the AI re-plan one scene ------------------------------------------
# await engine.regenerate_plan_for(project, project.scenes[2], prompt_prefix=PROMPT_PREFIX)

for i, sc in enumerate(project.scenes, 1):
    icon = '🔎' if sc.source == 'search' else '🎨'
    detail = sc.search_query if sc.source == 'search' else (sc.prompt or '')[:80]
    print(f'Scene {i}  {icon} {sc.source:8s} [{sc.plan_media_type}]  {detail}')


## 11. Step ③ — Search the stock providers (search scenes only)

Each search scene queries Pexels + Pixabay concurrently (responses are cached on disk for 24h), then the LLM ranks the candidates against the narration and auto-picks the best. If a search comes up empty, the scene **falls back to AI generation** automatically (a prompt is written for it).


In [ ]:
search_scenes = [(i, sc) for i, sc in enumerate(project.scenes) if sc.is_search]
print(f'{len(search_scenes)} search scene(s)')

for i, sc in search_scenes:
    if not sc.search_attempted:
        await engine.search_scene_media(project, sc, i, prompt_prefix=PROMPT_PREFIX)

for i, sc in enumerate(project.scenes):
    if sc.fell_back_to_generate:
        print(f"Scene {i+1}: ↩️ no stock results for '{sc.search_query}' — falls back to generation")
        print(f'   🎨 prompt: {sc.prompt}')
    elif sc.is_search:
        print(f"Scene {i+1}: 🔎 '{sc.search_query}' → {len(sc.candidates)} candidate(s)")
        for c in sc.candidates:
            mark = '🏆' if c.id == sc.picked_candidate_id else '  '
            print(f'   {mark} [{c.id}] {c.meta_line()}')
    print()


### Review the candidate galleries (thumbnails)

The AI pick is framed in green. To override it, set `picked_candidate_id` in the next cell.


In [ ]:
from IPython.display import HTML, display

for i, sc in enumerate(project.scenes):
    if not sc.is_search or not sc.candidates:
        continue
    tiles = []
    for c in sc.candidates:
        picked = c.id == sc.picked_candidate_id
        border = '4px solid #22c55e' if picked else '1px solid #999'
        badge = '🏆 AI pick · ' if picked else ''
        dur = f' · {c.duration:.0f}s' if c.duration else ''
        tiles.append(
            f'<div style="display:inline-block;margin:4px;text-align:center;max-width:170px;vertical-align:top">'
            f'<img src="{c.thumbnail_url}" style="width:160px;border:{border};border-radius:6px">'
            f'<br><small>{badge}{c.source} · {c.width}x{c.height}{dur}<br>{c.id}</small></div>'
        )
    display(HTML(f'<b>Scene {i+1}</b> — 🔎 <code>{sc.search_query}</code><br>' + ''.join(tiles)))


### Apply the picks (download + normalize)

Optionally override any pick first, then this cell downloads each picked candidate into the scene's canonical asset (`frames/<uid>_image.png` / `<uid>_video.mp4`), **re-encoded to the project size and fps** so stock clips concat cleanly with generated scenes.


In [ ]:
# --- Override a pick first (uncomment & adapt; use a candidate id from above) --
# sc = project.scenes[0]
# sc.picked_candidate_id = sc.candidates[2].id

from IPython.display import Image, Video, display

for i, sc in enumerate(project.scenes):
    if sc.is_search and sc.picked_candidate_id and not sc.has_media:
        await engine.apply_picked_candidate(project, sc, i)

for i, sc in enumerate(project.scenes):
    if sc.is_search and sc.attribution:
        a = sc.attribution
        print(f"Scene {i+1}: {sc.media_type} by {a.get('photographer') or 'unknown'} "
              f"({a.get('source')}, {a.get('license')})")
        if sc.media_type == 'video':
            display(Video(sc.video_path, embed=True, width=260))
        else:
            display(Image(sc.image_path, width=260))


## 12. Step ④ — Generate all scenes

`process_scene` runs whatever is still missing for each scene — audio, then **search → rank → download** for search scenes or **media generation** for generate scenes (incl. the fallback ones), then the subtitled segment. Re-run this cell safely — finished pieces are skipped.

> The **first** AI media generation downloads the model checkpoint (a few GB) — subsequent scenes reuse the model already loaded in VRAM. Stock-only videos skip the model download entirely.


In [ ]:
for i, sc in enumerate(project.scenes):
    if sc.segment_path:
        print(f'Scene {i+1}: already done ✅')
        continue
    icon = '🔎' if sc.is_search else '🎨'
    await engine.process_scene(
        project, sc, i,
        progress_callback=lambda stage, i=i: print(f'  Scene {i+1}: {stage}...'),
    )
    print(f'Scene {i+1} {icon}: done ✅ ({sc.duration:.1f}s)')

print()
print(f'Segments ready: {sum(1 for s in project.scenes if s.segment_path)}/{len(project.scenes)}')


### Review all segments (optional)


In [ ]:
from IPython.display import Video, display

for i, sc in enumerate(project.scenes, 1):
    icon = '🔎 stock' if (sc.attribution) else '🎨 generated'
    print(f'Scene {i} ({icon}): {sc.narration}')
    display(Video(sc.segment_path, embed=True, width=260))


### Not happy with a scene? Regenerate any piece

Edit and re-run only what changed — invalidation is automatic:


In [ ]:
# sc = project.scenes[0]; i = 0

# --- Change the narration (invalidates audio + segment; stock media survives) --
# sc.narration = 'A brand new opening line.'
# sc.invalidate_audio()
# await engine.generate_audio(project, sc, i)

# --- Search scene: different keywords -> re-search, re-pick, re-apply ----------
# sc.search_query = 'lighthouse storm waves'
# sc.invalidate_search(); sc.invalidate_media()
# await engine.search_scene_media(project, sc, i, prompt_prefix=PROMPT_PREFIX)
# await engine.apply_picked_candidate(project, sc, i)

# --- Generate scene: new prompt -> regenerate media ----------------------------
# sc.prompt = 'a paper boat on a dark sea under moonlight, ' + PROMPT_PREFIX
# sc.invalidate_media()
# await engine.generate_media(project, sc, i)

# --- Then re-render the segment ------------------------------------------------
# await engine.render_segment(project, sc, i)

for i, sc in enumerate(project.scenes, 1):
    print(f'Scene {i}:',
          'audio ✅' if sc.audio_path else 'audio ⬜',
          '| media ✅' if sc.has_media else '| media ⬜',
          '| segment ✅' if sc.segment_path else '| segment ⬜')


## 13. Step ⑤ — Compose the final video

Concatenates all segments and (optionally) adds background music, then persists the task (labeled `flexible_video`, with per-scene sourcing + attribution metadata) so it appears in the web UI's History page.


In [ ]:
BGM = None            # or e.g. 'default.mp3' (any file in Pixelle_video/bgm/)
BGM_VOLUME = 0.2

result = await engine.compose_final(project, bgm_path=BGM, bgm_volume=BGM_VOLUME)

print(f"Final video: {result['video_path']}")
print(f"Duration:    {result['duration']:.1f}s | Size: {result['file_size'] / 1e6:.1f} MB | Scenes: {result['n_scenes']}")

# --- Stock media credits (Pexels/Pixabay attribution duty) -------------------
credited = [(i, s) for i, s in enumerate(project.scenes, 1) if s.attribution]
if credited:
    print()
    print('📜 Stock media credits:')
    for i, sc in credited:
        a = sc.attribution
        print(f"  Scene {i}: {a.get('photographer') or 'unknown'} — {a.get('source')} "
              f"({a.get('license')})  {a.get('page_url') or ''}")


### Preview the result


In [ ]:
from IPython.display import Video

Video(result['video_path'], embed=True, width=320)


## 14. (Optional) Launch the Flexible Video Web UI

The same flow as a 6-step **Streamlit wizard** (Setup → Script → Media Plan → Source Media gallery → Scenes → Final + credits), exposed through a free Cloudflare quick tunnel. Click the printed `trycloudflare.com` link; keep the cell running while you use the UI and press **Stop** when done.

It shares the same config / output as this notebook, so videos composed in the UI also land in `Pixelle_video/output/` and show up in the History page.

> **If the page errors with `Failed to fetch dynamically imported module`:** do a **hard refresh** (`Ctrl+Shift+R` / `Cmd+Shift+R`) — the browser cached the UI of a previous run whose JS chunks no longer exist. If it persists, stop and re-run this cell to get a fresh tunnel (the free tunnel occasionally drops chunk requests; the cell already forces the more reliable http2 transport).


In [ ]:
import os, re, subprocess, sys

# Cloudflare quick tunnel binary
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/usr/local/bin/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

env = os.environ.copy()
env['PIXELLE_VIDEO_ROOT'] = str(PIXELLE_ROOT)
env['PYTHONPATH'] = f"{FLEX_ROOT}:{SBS_ROOT}:{WAN2GP_ROOT}:{PIXELLE_ROOT}"

streamlit_proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', str(FLEX_ROOT / 'web' / 'app.py'),
     '--server.port', '8504', '--server.headless', 'true',
     '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false',
     '--browser.gatherUsageStats', 'false'],
    cwd=str(PIXELLE_ROOT), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8504',
     '--protocol', 'http2', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

print('Waiting for the tunnel URL...')
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        print(f'\n🌐 Flexible Video Web UI: {match.group(0)}\n')
        break

try:
    for line in iter(streamlit_proc.stdout.readline, ''):
        if not line:
            break
        print(line, end='')
except KeyboardInterrupt:
    print('Stopping...')
finally:
    streamlit_proc.terminate()
    tunnel_proc.terminate()
    print('Web UI and tunnel stopped.')


## Notes & troubleshooting

- **Outputs** land in `Pixelle_video/output/<task_id>/final.mp4`; per-scene assets in `frames/<scene_uid>_*.{mp3,png,mp4}` (uid-based, so regenerating never collides).
- **Stock API quotas**: Pexels 200 req/hour + 20,000/month (free unlimited on request with visible attribution); Pixabay 100 req/60s. Search responses are **cached on disk for 24h** (`Pixelle_video_flexible/.search_cache/`, required by Pixabay's terms) and a 429 is retried once honoring `Retry-After`. If one provider is rate-limited or down, the other still serves results.
- **No hotlinking**: picked stock media is always downloaded into the task directory and re-encoded — Pixabay's returned URLs expire after ~24h, so never store them.
- **Mixed look**: a strong `PROMPT_PREFIX` (e.g. stick-figure sketch) makes generated scenes clash with real stock footage. Use a photographic prefix or none for mixed videos, or set all scenes to one source in the plan step.
- **Search finds nothing?** The engine first retries once with LLM-broadened keywords; if still empty, the scene falls back to AI generation (`allow_fallback: true`). You can also flip any scene to `generate` in the plan step.
- **Stock-only videos (no image/video generator at all)**: set `SEARCH_ONLY = True` in step 6 — the plan forces every scene to search, abstract ideas are rephrased into stock-findable imagery, and empty searches retry with broader keywords instead of generating. Since WanGP models load lazily on first generation, a stock-only run never downloads a model and **doesn't need the GPU** (TTS + ffmpeg + Chromium only — a CPU Colab runtime works).
- **Stock clip length** doesn't need to match the narration: the segment step trims long clips and freeze-pads short ones to the narration audio.
- **Concat safety**: stock media is re-encoded to the project size/fps (cover-crop, yuv420p, silent) before entering the segment — required because the final compose uses ffmpeg's `-c copy` concat.
- **Narration too quiet?** The engine loudness-normalizes every narration to -16 LUFS after TTS (`tts_normalize`, on by default) and can add extra gain (`tts_volume`, e.g. 1.5). Alternatively lower the BGM (`BGM_VOLUME`).
- **Regeneration rules**: changing a narration invalidates audio + segment (applied stock media *survives* — it isn't length-synced); changing keywords invalidates the search; changing a prompt invalidates generated media.
- **First AI generation is slow**: WanGP downloads the model checkpoint, then keeps it loaded in VRAM — later scenes are much faster. Stock-only videos need no model at all.
- **Out of VRAM / RAM on T4**: stick to `image_z_image` + `video_wan2.1_1.3B`, keep `--profile 5`, and use smaller media sizes (the template's media size is capped automatically by each descriptor's `max_pixels`).
- **Reasoning LLMs** (MiniMax-M3, DeepSeek-R1, Qwen3, ...): hidden chain-of-thought counts against the token budget, which can yield `LLM returned no content`. The flexible stages (script / plan / ranking) retry automatically with an escalating budget; if a step still fails, switch to a non-reasoning model (e.g. `deepseek-chat`).
- **`AttributeError: module 'pkgutil' has no attribute 'ImpImporter'`**: an old system `pkg_resources` is shadowing the modern one on Python 3.12. Run `pip install --upgrade setuptools wheel` and restart (the install cell now does this automatically).
- Full backend documentation: `Pixelle_video/WAN2GP_BACKEND.md` · app documentation: `Pixelle_video_flexible/README.md` (the web UI lives in `Pixelle_video_flexible/web/`, port 8504) · per-scene engine: `Pixelle_video_scene_by_scene/README.md`.
